This notebook verifies that final matched Bogard Lakes dataset doesn't have any mismatches with geocode info available

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np

from land_cover.load import loadBogardSuppl, bogard_output_path_raw, loadBogardMapShp

In [2]:
version = "v3"
_, out_dir, file_name = loadBogardSuppl()
out_spatial_stem = out_dir / "shp" / "qa_qc" / f"{file_name}_geocoded_{version}"
geocode_pth = f"{out_spatial_stem}.gpkg"
updated_gee_pth = "/Volumes/metis/ABOVE3/Tom/gee_input/updated/gee_cleaned_geocode_2025-07-10.csv"
print(geocode_pth)

/Volumes/metis/ABOVE3/Bogard_suppl_data/edk_out/shp/qa_qc/Bogard19_ESM_alldata_wh_geocoded_v3.gpkg


In [3]:
# Load gee table as DataFrame and convert to GeoDataFrame
gee_output_gdf = gpd.read_file(bogard_output_path_raw)

# Load geocode table as GeoDataFrame
geocode_gdf = gpd.read_file(geocode_pth)
len(gee_output_gdf)

# load bml shapefile for merging in index name from Tom
gdf_bogard_shp = loadBogardMapShp(ABOVE_region=False, region="WH")

In [20]:
gdf_bogard_shp.head()
gdf_bogard_shp.info()
gdf_bogard_shp.iloc[0,:]

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 45 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   lake_id        378 non-null    float64 
 1   Lake_area      378 non-null    float64 
 2   Outlet_n       378 non-null    int64   
 3   D_lake_id      337 non-null    object  
 4   D_lake_n       378 non-null    int64   
 5   D_lak_ntot     378 non-null    int64   
 6   U_lake_n       378 non-null    int64   
 7   U_lak_ntot     378 non-null    int64   
 8   Cat_a_lake     378 non-null    float64 
 9   Lake_type      378 non-null    object  
 10  Lake_order     378 non-null    int64   
 11  Laktyp_mhv     378 non-null    object  
 12  Lperm_glcp     378 non-null    float64 
 13  Basin_id       378 non-null    object  
 14  MERGE_SRC      378 non-null    object  
 15  Shape_Leng     378 non-null    float64 
 16  Shape_Area     378 non-null    float64 
 17  PLDNdistm_     0 non-null  

lake_id                                               7130662572.0
Lake_area                                                 1.106395
Outlet_n                                                         1
D_lake_id                                               7130665163
D_lake_n                                                         1
D_lak_ntot                                                       5
U_lake_n                                                         2
U_lak_ntot                                                       2
Cat_a_lake                                               23.643285
Lake_type                                             Flow Through
Lake_order                                                       2
Laktyp_mhv                                         InflowHeadwater
Lperm_glcp                                                  -99.99
Basin_id                                                 710000331
MERGE_SRC        D:\__THOWARD\ABOVE_Project\Technical\InputDat

In [6]:
gee_output_gdf.head()

,PointLat,PointLon,caption,pld_match,sampleUID,savetype,geometry
0,48.128496885028184,-71.28594971031188,No Caption,-99999.31415,BML_00000060,New polygon,"POLYGON ((-71.28625 48.13021, -71.28599 48.130..."
1,48.48063681142981,-79.4085016785392,tiny beaver pond- approximate,-99999.31415,BML_00000097,New polygon,"POLYGON ((-79.40864 48.48067, -79.40856 48.480..."
2,-99999.31415,-99999.31415,"shield, bad google imagery",-99999.31415,BML_00000281,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
3,-99999.31415,-99999.31415,closest lake to this road point,8223660592,BML_00005825,MatchedPLD,"POLYGON ((-133.05027 68.08287, -133.05025 68.0..."
4,-99999.31415,-99999.31415,also digitized,7240067712,BML_00000060,MatchedPLD,"POLYGON ((-71.28654 48.12908, -71.28574 48.129..."


In [7]:
# join to bofard shape lakes in WH
geocode_gdf = geocode_gdf.merge(
    gdf_bogard_shp, left_on=["lat (decimal)", "long (decimal)"], right_on=["lat", "long"]
)

In [8]:
geocode_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 356 entries, 0 to 355
Data columns (total 68 columns):
 #   Column                                      Non-Null Count  Dtype   
---  ------                                      --------------  -----   
 0   Category (1=same as YFB lakes/2=different)  356 non-null    float64 
 1   coscatsID                                   0 non-null      float64 
 2   Sbcode                                      0 non-null      float64 
 3   otherID provided                            343 non-null    object  
 4   lake name provided                          351 non-null    object  
 5   lat (decimal)                               356 non-null    float64 
 6   long (decimal)                              356 non-null    float64 
 7   pco2 (uatm)                                 353 non-null    float64 
 8   replicates (number)                         10 non-null     float64 
 9   CO2 analysis method                         356 non-null    object  

FYI, all these geocoded lakes are in eastern NA, not in ABOVE region

In [14]:
geocoded = geocode_gdf.query("geocoder.notnull()")
geocoded.head()

,Category (1=same as YFB lakes/2=different),coscatsID,Sbcode,otherID provided,lake name provided,lat (decimal),long (decimal),pco2 (uatm),replicates (number),CO2 analysis method,...,AvgOfALKum,AvgOfCO2_o,StDevOfCO2,AvgOfpCO2,StDevOfpCO,Name.1,PopupInfo,LabelID,Name,LitReviewInfo
74,2.0,NaN,NaN,L205,Lac Dolly,54.81829,-66.77029,414.0,NaN,equilibrator,...,NaN,NaN,NaN,414.0,NaN,None,None,NaN,BogardMapLakes,0
76,2.0,NaN,NaN,L207,Lac Rivals,54.86770,-66.67668,411.0,NaN,equilibrator,...,NaN,NaN,NaN,411.0,NaN,None,None,NaN,BogardMapLakes,0
78,2.0,NaN,NaN,L209,Lac Maryjo,54.81471,-66.79951,476.0,NaN,equilibrator,...,NaN,NaN,NaN,476.0,NaN,None,None,NaN,BogardMapLakes,0
79,2.0,NaN,NaN,L210,Lac Squaw,54.83115,-66.80691,487.0,NaN,equilibrator,...,NaN,NaN,NaN,487.0,NaN,None,None,NaN,BogardMapLakes,0
80,2.0,NaN,NaN,L211,Lac chantale,54.82891,-66.82272,406.0,NaN,equilibrator,...,NaN,NaN,NaN,406.0,NaN,None,None,NaN,BogardMapLakes,0


In [16]:
geocoded.lake_id

74     7.221554e+09
76     7.221566e+09
78     7.221554e+09
79     7.221554e+09
80     7.221554e+09
           ...     
345    7.240079e+09
346    7.240071e+09
347    7.211176e+09
348    7.240020e+09
349    7.240050e+09
Name: lake_id, Length: 118, dtype: float64

In [13]:
# gee_output_gdf.query("savetype == 'Mismatch'").set_index(["PointLat", "PointLon"]).index
gee_output_gdf.query("savetype == 'Mismatch'").set_index("sampleUID")

,PointLat,PointLon,caption,pld_match,savetype,geometry
sampleUID,,,,,,
BML_00000281,-99999.31415,-99999.31415,"shield, bad google imagery",-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
BML_00000000,-99999.31415,-99999.31415,antarctica,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
BML_00000282,-99999.31415,-99999.31415,No Caption,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
BML_00000297,-99999.31415,-99999.31415,No Caption,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
BML_00000312,-99999.31415,-99999.31415,Estuary,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
...,...,...,...,...,...,...
BML_00006194,-99999.31415,-99999.31415,No Caption,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
BML_00006181,-99999.31415,-99999.31415,No Caption,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
BML_00006182,-99999.31415,-99999.31415,No Caption,-99999.31415,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."


In [5]:
# First, set QA pass columns to not geocoded
geocode_fail_idx = geocode_gdf.query("QA == 0").index
geocode_gdf.loc[geocode_fail_idx, ["geocoder", "geocode_full_name", "geocode_lat", "geocode_lon"]] = (
    np.nan
)
geocode_gdf.loc[geocode_fail_idx, ["geocode_geom"]] = False

In [6]:
# How many matched features had no lat/lon? Actually, all have lat/lon
count_bogard_suppl_with_coords = geocode_gdf[
    geocode_gdf["lat (decimal)"].notnull() & geocode_gdf["long (decimal)"].notnull()
].shape[0]

count_geocode_no_coords = geocode_gdf[(geocode_gdf['geocoder'].notnull()) & (geocode_gdf['lat (decimal)'].isnull())].shape[0]
print(
    f"Number of bogard_suppl features with lat/lon: {count_bogard_suppl_with_coords} / {len(geocode_gdf)}"
)
print(
    f"Number of geocoded features with no lat/lon: {count_geocode_no_coords} / {len(geocode_gdf)}"
)

Number of bogard_suppl features with lat/lon: 922 / 922
Number of geocoded features with no lat/lon: 0 / 922
